<a href="https://colab.research.google.com/github/Vamkrishhh/Business-Entity-Resolution-Amazon-ML-Challenge/blob/main/Amazon_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL PIPELINE (runs on synthetic data)
# ============================================================

!pip install rapidfuzz

import os, re, math, random
import pandas as pd, numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

os.makedirs("/content/src/blocking", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

# ------------------------------------------------------------
# 1. SYNTHETIC DATA GENERATOR (mimics US / India / France)
# ------------------------------------------------------------
def make_synthetic_dataset(n_s1=400, seed=42, noise=0.35):
    random.seed(seed); np.random.seed(seed)
    COUNTRIES = ["US", "India", "France"]
    NAMES = ["Acme Corporation", "Global Tech Pvt Ltd", "Bright Star LLC",
             "Sunrise Foods", "Metro Logistics", "Blue Ocean Traders",
             "Pinnacle Software", "Delta Enterprises", "Green Leaf Cafe",
             "Nova Pharma", "Silverline Mfg", "Orbit Distributors",
             "Lumiere SARL", "Atelier du Bois", "Maison Verte SAS"]
    CITIES = ["Springfield", "Mumbai", "Bengaluru", "Paris", "Lyon",
              "Marseille", "Pune", "Austin", "Denver"]
    STREETS = ["Main Street", "MG Road", "Rue Victor Hugo", "Park Avenue",
               "Nehru Nagar", "Avenue de la Republique", "Elm Street"]

    def mutate_name(n):
        if random.random() > noise: return n
        op = random.choice(["abbr","typo","order","punct"])
        if op == "abbr":
            n = (n.replace("Corporation","Corp").replace("Private Limited","Pvt Ltd")
                   .replace("Limited","Ltd"))
        elif op == "typo" and len(n) > 5:
            i = random.randrange(1, len(n)-1); n = n[:i] + n[i+1:]
        elif op == "order" and " " in n:
            p = n.split(); random.shuffle(p); n = " ".join(p)
        elif op == "punct":
            n = n.replace(" ", random.choice(["-","."," "]))
        return n

    def mutate_addr(a):
        if random.random() > noise: return a
        op = random.choice(["drop_pin","abbr","reorder"])
        if op == "drop_pin":
            a = re.sub(r"\b\d{5,6}\b","",a).strip().strip(",")
        elif op == "abbr":
            a = (a.replace("Street","St").replace("Road","Rd").replace("Avenue","Ave"))
        elif op == "reorder":
            parts = [p.strip() for p in a.split(",") if p.strip()]
            random.shuffle(parts); a = ", ".join(parts)
        return a

    def rand_addr():
        num = random.randint(1, 999)
        return f"{num} {random.choice(STREETS)}, {random.choice(CITIES)} " \
               f"{random.randint(100000,999999)}"

    s1_rows, s2_rows, s3_rows, gt_rows = [], [], [], []
    for i in range(n_s1):
        sid = f"S1-{i:05d}"
        bn = random.choice(NAMES); ba = rand_addr(); bc = random.choice(COUNTRIES)
        s1_rows.append({"entity_id": sid, "business_name": bn,
                        "business_address": ba, "country": bc})
        r = random.random()
        if   r < 0.20: n2, n3 = 0, 0
        elif r < 0.55: n2, n3 = 1, 0
        elif r < 0.75: n2, n3 = 0, 1
        else:          n2, n3 = random.randint(1,2), random.randint(1,2)
        matched = []
        for _ in range(n2):
            eid = f"S2-{len(s2_rows):05d}"
            s2_rows.append({"entity_id": eid, "business_name": mutate_name(bn),
                            "business_address": mutate_addr(ba), "country": bc})
            matched.append(eid)
        for _ in range(n3):
            eid = f"S3-{len(s3_rows):05d}"
            s3_rows.append({"entity_id": eid, "business_name": mutate_name(bn),
                            "business_address": mutate_addr(ba), "country": bc})
            matched.append(eid)
        gt_rows.append({"source1_entity_id": sid, "matched_entity_ids": ",".join(matched)})

    for _ in range(int(n_s1*1.5)):
        s2_rows.append({"entity_id": f"S2-{len(s2_rows):05d}",
                        "business_name": mutate_name(random.choice(NAMES)),
                        "business_address": rand_addr(),
                        "country": random.choice(COUNTRIES)})
    for _ in range(int(n_s1*1.5)):
        s3_rows.append({"entity_id": f"S3-{len(s3_rows):05d}",
                        "business_name": mutate_name(random.choice(NAMES)),
                        "business_address": rand_addr(),
                        "country": random.choice(COUNTRIES)})

    return (pd.DataFrame(s1_rows), pd.DataFrame(s2_rows),
            pd.DataFrame(s3_rows), pd.DataFrame(gt_rows))

# ------------------------------------------------------------
# 2. NORMALIZATION (name + address)
# ------------------------------------------------------------
ABBR = {"corp":"corporation","co":"company","ltd":"limited",
        "llc":"limited liability company","pvt":"private","inc":"incorporated",
        "rd":"road","st":"street","ave":"avenue","blvd":"boulevard",
        "dr":"drive","ln":"lane","ct":"court","hwy":"highway",
        "mfg":"manufacturing","intl":"international","svc":"service",
        "svcs":"services","ent":"enterprise","grp":"group",
        "assoc":"association","bros":"brothers","tech":"technology",
        "ind":"industries","pharm":"pharmaceutical"}

LEGAL = {"corporation","company","limited","limited liability company",
         "private","incorporated","llp","plc","sarl","sas","gmbh"}
STOPW = {"the","of","and","for","at","in","on","a","an"}

def normalize(s):
    if s is None or (isinstance(s,float) and math.isnan(s)): return ""
    s = str(s).lower().replace("&"," and ")
    s = re.sub(r"[^a-z0-9\s]"," ",s)
    return " ".join(ABBR.get(t,t) for t in s.split()).strip()

def name_toks(s):
    return [t for t in normalize(s).split()
            if len(t)>=3 and t not in STOPW and t not in LEGAL]

ADDR_STOP = {"road","street","avenue","lane","drive","near","opposite",
             "floor","suite","building","nagar","colony","sector","block",
             "main","cross","india","usa","united","states","france","paris",
             "post","po","district","state","pin","code","zip","area"}

def _digits(s): return "" if s is None else re.sub(r"\D","",str(s))

def postal(s):
    d = _digits(s)
    if len(d)>=6: return d[-6:]
    if len(d)>=5: return d[-5:]
    return ""

def street_num(s):
    m = re.search(r"\b(\d{1,6})\b", str(s))
    return m.group(1) if m else ""

def addr_toks(s):
    n = re.sub(r"[^a-z0-9\s]"," ",str(s).lower())
    return [t for t in n.split()
            if len(t)>=4 and t not in ADDR_STOP and not t.isdigit()]

def city_toks(s):
    n = re.sub(r"[^a-z0-9\s]"," ",str(s).lower())
    return [t for t in n.split()
            if len(t)>=5 and t not in ADDR_STOP and not t.isdigit()]

def prepare(df):
    df = df.copy()
    for c in ("business_name","business_address","country"):
        if c not in df.columns: df[c] = ""
    df["norm_name"]  = df["business_name"].fillna("").apply(normalize)
    df["norm_addr"]  = df["business_address"].fillna("").apply(normalize)
    df["name_tokens"]= df["norm_name"].apply(name_toks)
    df["addr_tokens"]= df["business_address"].fillna("").apply(addr_toks)
    df["city_tokens"]= df["business_address"].fillna("").apply(city_toks)
    df["postal"]     = df["business_address"].fillna("").apply(postal)
    df["street_num"] = df["business_address"].fillna("").apply(street_num)
    return df

# ------------------------------------------------------------
# 3. BLOCKING INDEXES
# ------------------------------------------------------------
def build_name_indexes(df):
    exact, prefix, token, sortedk = (defaultdict(set) for _ in range(4))
    for _, r in df.iterrows():
        eid = r["entity_id"]; n = r["norm_name"]
        if not n: continue
        exact[n].add(eid)
        if len(n)>=4: prefix[n[:4]].add(eid)
        toks = r["name_tokens"]
        for t in toks: token[t].add(eid)
        if toks: sortedk[" ".join(sorted(toks))].add(eid)
    return {"exact":exact,"prefix":prefix,"token":token,"sorted":sortedk}

def build_addr_indexes(df):
    post, street, city, tok = (defaultdict(set) for _ in range(4))
    for _, r in df.iterrows():
        eid = r["entity_id"]
        if r["postal"]:     post[r["postal"]].add(eid)
        if r["street_num"]: street[r["street_num"]].add(eid)
        for c in r["city_tokens"]: city[c].add(eid)
        for t in r["addr_tokens"]: tok[t].add(eid)
    return {"postal":post,"street":street,"city":city,"token":tok}

def build_combined_indexes(df):
    np_, nc, na = (defaultdict(set) for _ in range(3))
    for _, r in df.iterrows():
        eid = r["entity_id"]; p = r["postal"]
        for t in r["name_tokens"]:
            if p: np_[(t,p)].add(eid)
            for c in r["city_tokens"]: nc[(t,c)].add(eid)
            for a in r["addr_tokens"][:6]: na[(t,a)].add(eid)
    return {"name_postal":np_,"name_city":nc,"name_addrtok":na}

def name_cands(row, idx):
    out = set(); n = row["norm_name"]
    if not n: return out
    out |= idx["exact"].get(n,set())
    if len(n)>=4: out |= idx["prefix"].get(n[:4],set())
    for t in row["name_tokens"]: out |= idx["token"].get(t,set())
    if row["name_tokens"]:
        out |= idx["sorted"].get(" ".join(sorted(row["name_tokens"])),set())
    return out

def addr_cands(row, idx):
    out = set()
    if row["postal"]:     out |= idx["postal"].get(row["postal"],set())
    if row["street_num"]: out |= idx["street"].get(row["street_num"],set())
    for c in row["city_tokens"]: out |= idx["city"].get(c,set())
    for t in row["addr_tokens"][:6]: out |= idx["token"].get(t,set())
    return out

def comb_cands(row, idx):
    out = set(); p = row["postal"]
    for t in row["name_tokens"]:
        if p: out |= idx["name_postal"].get((t,p),set())
        for c in row["city_tokens"]: out |= idx["name_city"].get((t,c),set())
        for a in row["addr_tokens"][:6]: out |= idx["name_addrtok"].get((t,a),set())
    return out

# ------------------------------------------------------------
# 4. FUZZY + TF-IDF
# ------------------------------------------------------------
from rapidfuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer

def fuzzy_cands(s1, others, top_k=20, cutoff=65):
    other_ids = others["entity_id"].tolist()
    other_names = others["norm_name"].tolist()
    out = {}
    for _, r in s1.iterrows():
        sid = r["entity_id"]; q = r["norm_name"]
        if not q: out[sid] = set(); continue
        m = process.extract(q, other_names, scorer=fuzz.token_set_ratio, limit=top_k)
        out[sid] = {other_ids[i] for _, sc, i in m if sc >= cutoff}
    return out

def tfidf_cands(s1, others, top_k=20, cutoff=0.20):
    if len(others)==0: return {sid:set() for sid in s1["entity_id"]}
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), min_df=1)
    Xo = vec.fit_transform((others["norm_name"] + " " + others["norm_addr"]).tolist())
    Xs = vec.transform((s1["norm_name"] + " " + s1["norm_addr"]).tolist())
    oids = others["entity_id"].tolist(); out = {}
    for i, sid in enumerate(s1["entity_id"]):
        sims = (Xs[i] @ Xo.T).toarray().ravel()
        if len(sims)==0: out[sid]=set(); continue
        k = min(top_k, len(sims))
        ti = np.argpartition(-sims, k-1)[:k]
        ti = ti[np.argsort(-sims[ti])]
        out[sid] = {oids[j] for j in ti if sims[j] >= cutoff}
    return out

# ------------------------------------------------------------
# 5. GENERATE CANDIDATES
# ------------------------------------------------------------
def generate(s1, others, nidx, aidx, cidx, top_k=20, fcut=65, tcut=0.20):
    valid = set(others["entity_id"])
    structural = {}
    for _, r in s1.iterrows():
        sid = r["entity_id"]
        structural[sid] = name_cands(r,nidx) | addr_cands(r,aidx) | comb_cands(r,cidx)
    fuzzy = fuzzy_cands(s1, others, top_k, fcut)
    tfidf = tfidf_cands(s1, others, top_k, tcut)
    final = {}
    for sid in s1["entity_id"]:
        s = (structural[sid] | fuzzy.get(sid,set()) | tfidf.get(sid,set())) & valid
        final[sid] = s
    return final

# ------------------------------------------------------------
# 6. RUN ON TRAINING
# ------------------------------------------------------------
print("Generating synthetic training data...")
train_s1, train_s2, train_s3, gt = make_synthetic_dataset(n_s1=400, seed=42)
train_s1 = prepare(train_s1); train_s2 = prepare(train_s2); train_s3 = prepare(train_s3)
others = pd.concat([train_s2, train_s3], ignore_index=True)

print(f"  S1: {train_s1.shape}   Others: {others.shape}   GT: {gt.shape}")

print("\nBuilding indexes...")
nidx = build_name_indexes(others)
aidx = build_addr_indexes(others)
cidx = build_combined_indexes(others)

print("Generating candidates...")
cands = generate(train_s1, others, nidx, aidx, cidx,
                 top_k=20, fcut=65, tcut=0.20)

sizes = np.array([len(v) for v in cands.values()])

# ------------------------------------------------------------
# 7. CANDIDATE RECALL
# ------------------------------------------------------------
total_true = hit = 0; per_ent = []
for _, r in gt.iterrows():
    sid = r["source1_entity_id"]; raw = r["matched_entity_ids"]
    tids = set()
    if pd.notna(raw) and str(raw).strip():
        tids = {x.strip() for x in str(raw).split(",") if x.strip()}
    tids = {x for x in tids if x.startswith(("S2-","S3-"))}
    c = cands.get(sid, set()); inter = tids & c
    total_true += len(tids); hit += len(inter)
    if tids: per_ent.append(len(inter)/len(tids))

micro = hit/total_true if total_true else 1.0
macro = float(np.mean(per_ent)) if per_ent else 1.0

print("\n" + "="*55)
print("MEMBER 2 — BLOCKING QUALITY REPORT")
print("="*55)
print(f"Total true matches         : {total_true}")
print(f"True matches in candidates : {hit}")
print(f"Candidate Recall (micro)   : {micro:.4f}")
print(f"Candidate Recall (macro)   : {macro:.4f}")
print(f"Avg candidates / S1        : {sizes.mean():.2f}")
print(f"Median candidates / S1     : {np.median(sizes):.0f}")
print(f"Max candidates / S1        : {sizes.max()}")
print(f"Min candidates / S1        : {sizes.min()}")
print(f"S1 with 0 candidates       : {(sizes==0).sum()}")
print("="*55)

# ------------------------------------------------------------
# 8. RUN ON TEST + WRITE candidate_pairs.tsv
# ------------------------------------------------------------
print("\nGenerating synthetic test data...")
test_s1, test_s2, test_s3, _ = make_synthetic_dataset(n_s1=200, seed=777)
test_s1 = prepare(test_s1); test_s2 = prepare(test_s2); test_s3 = prepare(test_s3)
test_others = pd.concat([test_s2, test_s3], ignore_index=True)

tnidx = build_name_indexes(test_others)
taidx = build_addr_indexes(test_others)
tcidx = build_combined_indexes(test_others)

print("Generating test candidates...")
test_cands = generate(test_s1, test_others, tnidx, taidx, tcidx,
                      top_k=20, fcut=65, tcut=0.20)

rows = [[sid, ",".join(sorted(test_cands.get(sid, set())))]
        for sid in test_s1["entity_id"]]
out = pd.DataFrame(rows, columns=["source1_entity_id","candidate_entity_ids"])
out.to_csv("/content/output/candidate_pairs.tsv", sep="\t", index=False)

print(f"\n✅ Wrote /content/output/candidate_pairs.tsv ({len(out)} rows)")
print("\nFirst 5 rows:")
print(out.head().to_string(index=False))

# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------
df = pd.read_csv("/content/output/candidate_pairs.tsv",
                 sep="\t", dtype=str, keep_default_na=False)
issues = []
if list(df.columns) != ["source1_entity_id","candidate_entity_ids"]:
    issues.append(f"Bad columns: {df.columns.tolist()}")
if df["source1_entity_id"].duplicated().any():
    issues.append("Duplicate S1 rows")
expected = set(test_s1["entity_id"]); got = set(df["source1_entity_id"])
if expected != got:
    issues.append(f"Missing {len(expected-got)}, extra {len(got-expected)}")
valid_ids = set(test_others["entity_id"])
for _, r in df.iterrows():
    raw = r["candidate_entity_ids"]
    if not raw: continue
    ids = raw.split(",")
    if len(ids) != len(set(ids)):
        issues.append(f"Dup IDs in {r['source1_entity_id']}")
    for x in ids:
        if not (x.startswith("S2-") or x.startswith("S3-")):
            issues.append(f"Bad prefix {x}")
        elif x not in valid_ids:
            issues.append(f"Unknown ID {x}")

print("\n" + "="*55)
print("VALIDATION:", "✅ PASS" if not issues else "❌ FAIL")
print("="*55)
for m in issues[:10]: print("  -", m)
if not issues:
    print(f"  Rows              : {len(df)}")
    print(f"  Non-empty         : {(df['candidate_entity_ids'] != '').sum()}")
    print(f"  Empty (singletons): {(df['candidate_entity_ids'] == '').sum()}")

# ------------------------------------------------------------
# 10. DOWNLOAD
# ------------------------------------------------------------
from google.colab import files
files.download("/content/output/candidate_pairs.tsv")
print("\n✅ Pipeline complete. File downloaded.")

Generating synthetic training data...
  S1: (400, 11)   Others: (1713, 11)   GT: (400, 2)

Building indexes...
Generating candidates...

MEMBER 2 — BLOCKING QUALITY REPORT
Total true matches         : 513
True matches in candidates : 513
Candidate Recall (micro)   : 1.0000
Candidate Recall (macro)   : 1.0000
Avg candidates / S1        : 401.71
Median candidates / S1     : 476
Max candidates / S1        : 551
Min candidates / S1        : 94
S1 with 0 candidates       : 0

Generating synthetic test data...
Generating test candidates...

✅ Wrote /content/output/candidate_pairs.tsv (200 rows)

First 5 rows:
source1_entity_id                                                                                                                                                                                                                                                                                                                                                                                    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Pipeline complete. File downloaded.


### Pipeline Setup

This section handles initial package installations, imports, and directory creation.

In [ ]:
# ============================================================
# MEMBER 2 — FULL PIPELINE (runs on synthetic data)
# ============================================================

!pip install rapidfuzz

import os, re, math, random
import pandas as pd, numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

os.makedirs("/content/src/blocking", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

### 1. Synthetic Data Generator

This function creates synthetic datasets that mimic real-world business data from different countries (US, India, France) with added noise for various entity resolution scenarios.

In [ ]:
# ------------------------------------------------------------
# 1. SYNTHETIC DATA GENERATOR (mimics US / India / France)
# ------------------------------------------------------------
def make_synthetic_dataset(n_s1=400, seed=42, noise=0.35):
    random.seed(seed); np.random.seed(seed)
    COUNTRIES = ["US", "India", "France"]
    NAMES = ["Acme Corporation", "Global Tech Pvt Ltd", "Bright Star LLC",
             "Sunrise Foods", "Metro Logistics", "Blue Ocean Traders",
             "Pinnacle Software", "Delta Enterprises", "Green Leaf Cafe",
             "Nova Pharma", "Silverline Mfg", "Orbit Distributors",
             "Lumiere SARL", "Atelier du Bois", "Maison Verte SAS"]
    CITIES = ["Springfield", "Mumbai", "Bengaluru", "Paris", "Lyon",
              "Marseille", "Pune", "Austin", "Denver"]
    STREETS = ["Main Street", "MG Road", "Rue Victor Hugo", "Park Avenue",
               "Nehru Nagar", "Avenue de la Republique", "Elm Street"]

    def mutate_name(n):
        if random.random() > noise: return n
        op = random.choice(["abbr","typo","order","punct"])
        if op == "abbr":
            n = (n.replace("Corporation","Corp").replace("Private Limited","Pvt Ltd")
                   .replace("Limited","Ltd"))
        elif op == "typo" and len(n) > 5:
            i = random.randrange(1, len(n)-1); n = n[:i] + n[i+1:]
        elif op == "order" and " " in n:
            p = n.split(); random.shuffle(p); n = " ".join(p)
        elif op == "punct":
            n = n.replace(" ", random.choice(["-","."," "]))
        return n

    def mutate_addr(a):
        if random.random() > noise: return a
        op = random.choice(["drop_pin","abbr","reorder"])
        if op == "drop_pin":
            a = re.sub(r"\b\d{5,6}\b","",a).strip().strip(",")
        elif op == "abbr":
            a = (a.replace("Street","St").replace("Road","Rd").replace("Avenue","Ave"))
        elif op == "reorder":
            parts = [p.strip() for p in a.split(",") if p.strip()]
            random.shuffle(parts); a = ", ".join(parts)
        return a

    def rand_addr():
        num = random.randint(1, 999)
        return f"{num} {random.choice(STREETS)}, {random.choice(CITIES)} " \
               f"{random.randint(100000,999999)}"

    s1_rows, s2_rows, s3_rows, gt_rows = [], [], [], []
    for i in range(n_s1):
        sid = f"S1-{i:05d}"
        bn = random.choice(NAMES); ba = rand_addr(); bc = random.choice(COUNTRIES)
        s1_rows.append({"entity_id": sid, "business_name": bn,
                        "business_address": ba, "country": bc})
        r = random.random()
        if   r < 0.20: n2, n3 = 0, 0
        elif r < 0.55: n2, n3 = 1, 0
        elif r < 0.75: n2, n3 = 0, 1
        else:          n2, n3 = random.randint(1,2), random.randint(1,2)
        matched = []
        for _ in range(n2):
            eid = f"S2-{len(s2_rows):05d}"
            s2_rows.append({"entity_id": eid, "business_name": mutate_name(bn),
                            "business_address": mutate_addr(ba), "country": bc})
            matched.append(eid)
        for _ in range(n3):
            eid = f"S3-{len(s3_rows):05d}"
            s3_rows.append({"entity_id": eid, "business_name": mutate_name(bn),
                            "business_address": mutate_addr(ba), "country": bc})
            matched.append(eid)
        gt_rows.append({"source1_entity_id": sid, "matched_entity_ids": ",".join(matched)})

    for _ in range(int(n_s1*1.5)):
        s2_rows.append({"entity_id": f"S2-{len(s2_rows):05d}",
                        "business_name": mutate_name(random.choice(NAMES)),
                        "business_address": rand_addr(),
                        "country": random.choice(COUNTRIES)})
    for _ in range(int(n_s1*1.5)):
        s3_rows.append({"entity_id": f"S3-{len(s3_rows):05d}",
                        "business_name": mutate_name(random.choice(NAMES)),
                        "business_address": rand_addr(),
                        "country": random.choice(COUNTRIES)})

    return (pd.DataFrame(s1_rows), pd.DataFrame(s2_rows),
            pd.DataFrame(s3_rows), pd.DataFrame(gt_rows))

### 2. Normalization Functions

This section defines functions to normalize business names and addresses, standardizing them for better matching. It handles abbreviations, removes punctuation, and extracts relevant tokens like postal codes and street numbers.

In [ ]:
# ------------------------------------------------------------
# 2. NORMALIZATION (name + address)
# ------------------------------------------------------------
ABBR = {"corp":"corporation","co":"company","ltd":"limited",
        "llc":"limited liability company","pvt":"private","inc":"incorporated",
        "rd":"road","st":"street","ave":"avenue","blvd":"boulevard",
        "dr":"drive","ln":"lane","ct":"court","hwy":"highway",
        "mfg":"manufacturing","intl":"international","svc":"service",
        "svcs":"services","ent":"enterprise","grp":"group",
        "assoc":"association","bros":"brothers","tech":"technology",
        "ind":"industries","pharm":"pharmaceutical"}

LEGAL = {"corporation","company","limited","limited liability company",
         "private","incorporated","llp","plc","sarl","sas","gmbh"}
STOPW = {"the","of","and","for","at","in","on","a","an"}

def normalize(s):
    if s is None or (isinstance(s,float) and math.isnan(s)): return ""
    s = str(s).lower().replace("&"," and ")
    s = re.sub(r"[^a-z0-9\s]"," ",s)
    return " ".join(ABBR.get(t,t) for t in s.split()).strip()

def name_toks(s):
    return [t for t in normalize(s).split()
            if len(t)>=3 and t not in STOPW and t not in LEGAL]

ADDR_STOP = {"road","street","avenue","lane","drive","near","opposite",
             "floor","suite","building","nagar","colony","sector","block",
             "main","cross","india","usa","united","states","france","paris",
             "post","po","district","state","pin","code","zip","area"}

def _digits(s): return "" if s is None else re.sub(r"\D","",str(s))

def postal(s):
    d = _digits(s)
    if len(d)>=6: return d[-6:]
    if len(d)>=5: return d[-5:]
    return ""

def street_num(s):
    m = re.search(r"\b(\d{1,6})\b", str(s))
    return m.group(1) if m else ""

def addr_toks(s):
    n = re.sub(r"[^a-z0-9\s]"," ",str(s).lower())
    return [t for t in n.split()
            if len(t)>=4 and t not in ADDR_STOP and not t.isdigit()]

def city_toks(s):
    n = re.sub(r"[^a-z0-9\s]"," ",str(s).lower())
    return [t for t in n.split()
            if len(t)>=5 and t not in ADDR_STOP and not t.isdigit()]

def prepare(df):
    df = df.copy()
    for c in ("business_name","business_address","country"):
        if c not in df.columns: df[c] = ""
    df["norm_name"]  = df["business_name"].fillna("").apply(normalize)
    df["norm_addr"]  = df["business_address"].fillna("").apply(normalize)
    df["name_tokens"]= df["norm_name"].apply(name_toks)
    df["addr_tokens"]= df["business_address"].fillna("").apply(addr_toks)
    df["city_tokens"]= df["business_address"].fillna("").apply(city_toks)
    df["postal"]     = df["business_address"].fillna("").apply(postal)
    df["street_num"] = df["business_address"].fillna("").apply(street_num)
    return df

### 3. Blocking Indexes

These functions create various indexes based on normalized names, addresses, and combined features to efficiently narrow down potential matches during the blocking phase. This reduces the number of comparisons needed significantly.

In [ ]:
# ------------------------------------------------------------
# 3. BLOCKING INDEXES
# ------------------------------------------------------------
def build_name_indexes(df):
    exact, prefix, token, sortedk = (defaultdict(set) for _ in range(4))
    for _, r in df.iterrows():
        eid = r["entity_id"]; n = r["norm_name"]
        if not n: continue
        exact[n].add(eid)
        if len(n)>=4: prefix[n[:4]].add(eid)
        toks = r["name_tokens"]
        for t in toks: token[t].add(eid)
        if toks: sortedk[" ".join(sorted(toks))].add(eid)
    return {"exact":exact,"prefix":prefix,"token":token,"sorted":sortedk}

def build_addr_indexes(df):
    post, street, city, tok = (defaultdict(set) for _ in range(4))
    for _, r in df.iterrows():
        eid = r["entity_id"]
        if r["postal"]:     post[r["postal"]].add(eid)
        if r["street_num"]: street[r["street_num"]].add(eid)
        for c in r["city_tokens"]: city[c].add(eid)
        for t in r["addr_tokens"]: tok[t].add(eid)
    return {"postal":post,"street":street,"city":city,"token":tok}

def build_combined_indexes(df):
    np_, nc, na = (defaultdict(set) for _ in range(3))
    for _, r in df.iterrows():
        eid = r["entity_id"]; p = r["postal"]
        for t in r["name_tokens"]:
            if p: np_[(t,p)].add(eid)
            for c in r["city_tokens"]: nc[(t,c)].add(eid)
            for a in r["addr_tokens"][:6]: na[(t,a)].add(eid)
    return {"name_postal":np_,"name_city":nc,"name_addrtok":na}

def name_cands(row, idx):
    out = set(); n = row["norm_name"]
    if not n: return out
    out |= idx["exact"].get(n,set())
    if len(n)>=4: out |= idx["prefix"].get(n[:4],set())
    for t in row["name_tokens"]: out |= idx["token"].get(t,set())
    if row["name_tokens"]:
        out |= idx["sorted"].get(" ".join(sorted(row["name_tokens"])) ,set())
    return out

def addr_cands(row, idx):
    out = set()
    if row["postal"]:     out |= idx["postal"].get(row["postal"],set())
    if row["street_num"]: out |= idx["street"].get(row["street_num"],set())
    for c in row["city_tokens"]: out |= idx["city"].get(c,set())
    for t in row["addr_tokens"][:6]: out |= idx["token"].get(t,set())
    return out

def comb_cands(row, idx):
    out = set(); p = row["postal"]
    for t in row["name_tokens"]:
        if p: out |= idx["name_postal"].get((t,p),set())
        for c in row["city_tokens"]: out |= idx["name_city"].get((t,c),set())
        for a in row["addr_tokens"][:6]: out |= idx["name_addrtok"].get((t,a),set())
    return out

### 4. Fuzzy and TF-IDF Matching

This section implements fuzzy string matching (using `rapidfuzz`) and TF-IDF similarity to identify potential matches between entities based on their names and addresses. These methods help to catch variations that exact matching might miss.

In [ ]:
# ------------------------------------------------------------
# 4. FUZZY + TF-IDF
# ------------------------------------------------------------
from rapidfuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer

def fuzzy_cands(s1, others, top_k=20, cutoff=65):
    other_ids = others["entity_id"].tolist()
    other_names = others["norm_name"].tolist()
    out = {}
    for _, r in s1.iterrows():
        sid = r["entity_id"]; q = r["norm_name"]
        if not q: out[sid] = set(); continue
        m = process.extract(q, other_names, scorer=fuzz.token_set_ratio, limit=top_k)
        out[sid] = {other_ids[i] for _, sc, i in m if sc >= cutoff}
    return out

def tfidf_cands(s1, others, top_k=20, cutoff=0.20):
    if len(others)==0: return {sid:set() for sid in s1["entity_id"]}
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), min_df=1)
    Xo = vec.fit_transform((others["norm_name"] + " " + others["norm_addr"]).tolist())
    Xs = vec.transform((s1["norm_name"] + " " + s1["norm_addr"]).tolist())
    oids = others["entity_id"].tolist(); out = {}
    for i, sid in enumerate(s1["entity_id"]):
        sims = (Xs[i] @ Xo.T).toarray().ravel()
        if len(sims)==0: out[sid]=set(); continue
        k = min(top_k, len(sims))
        ti = np.argpartition(-sims, k-1)[:k]
        ti = ti[np.argsort(-sims[ti])]
        out[sid] = {oids[j] for j in ti if sims[j] >= cutoff}
    return out

### 5. Generate Candidates

This function combines results from structural, fuzzy, and TF-IDF blocking methods to generate a comprehensive set of candidate matches for each entity. It acts as the core of the blocking strategy.

In [ ]:
# ------------------------------------------------------------
# 5. GENERATE CANDIDATES
# ------------------------------------------------------------
def generate(s1, others, nidx, aidx, cidx, top_k=20, fcut=65, tcut=0.20):
    valid = set(others["entity_id"])
    structural = {}
    for _, r in s1.iterrows():
        sid = r["entity_id"]
        structural[sid] = name_cands(r,nidx) | addr_cands(r,aidx) | comb_cands(r,cidx)
    fuzzy = fuzzy_cands(s1, others, top_k, fcut)
    tfidf = tfidf_cands(s1, others, top_k, tcut)
    final = {}
    for sid in s1["entity_id"]:
        s = (structural[sid] | fuzzy.get(sid,set()) | tfidf.get(sid,set())) & valid
        final[sid] = s
    return final

### 6. Run on Training Data

This section generates synthetic training data, prepares it using the normalization functions, builds indexes, and then generates candidate matches. This is used to evaluate the blocking quality.

In [ ]:
# ------------------------------------------------------------
# 6. RUN ON TRAINING
# ------------------------------------------------------------
print("Generating synthetic training data...")
train_s1, train_s2, train_s3, gt = make_synthetic_dataset(n_s1=400, seed=42)
train_s1 = prepare(train_s1); train_s2 = prepare(train_s2); train_s3 = prepare(train_s3)
others = pd.concat([train_s2, train_s3], ignore_index=True)

print(f"  S1: {train_s1.shape}   Others: {others.shape}   GT: {gt.shape}")

print("\nBuilding indexes...")
nidx = build_name_indexes(others)
aidx = build_addr_indexes(others)
cidx = build_combined_indexes(others)

print("Generating candidates...")
cands = generate(train_s1, others, nidx, aidx, cidx,
                 top_k=20, fcut=65, tcut=0.20)

sizes = np.array([len(v) for v in cands.values()])

Generating synthetic training data...
  S1: (400, 11)   Others: (1713, 11)   GT: (400, 2)

Building indexes...
Generating candidates...


### 7. Candidate Recall Report

This section evaluates the quality of the blocking by calculating the micro and macro recall of the generated candidates against the ground truth. It also provides statistics on the number of candidates per entity.

In [ ]:
# ------------------------------------------------------------
# 7. CANDIDATE RECALL
# ------------------------------------------------------------
total_true = hit = 0; per_ent = []
for _, r in gt.iterrows():
    sid = r["source1_entity_id"]; raw = r["matched_entity_ids"]
    tids = set()
    if pd.notna(raw) and str(raw).strip():
        tids = {x.strip() for x in str(raw).split(",") if x.strip()}
    tids = {x for x in tids if x.startswith(("S2-","S3-"))}
    c = cands.get(sid, set()); inter = tids & c
    total_true += len(tids); hit += len(inter)
    if tids: per_ent.append(len(inter)/len(tids))

micro = hit/total_true if total_true else 1.0
macro = float(np.mean(per_ent)) if per_ent else 1.0

print("\n" + "="*55)
print("MEMBER 2 — BLOCKING QUALITY REPORT")
print("="*55)
print(f"Total true matches         : {total_true}")
print(f"True matches in candidates : {hit}")
print(f"Candidate Recall (micro)   : {micro:.4f}")
print(f"Candidate Recall (macro)   : {macro:.4f}")
print(f"Avg candidates / S1        : {sizes.mean():.2f}")
print(f"Median candidates / S1     : {np.median(sizes):.0f}")
print(f"Max candidates / S1        : {sizes.max()}")
print(f"Min candidates / S1        : {sizes.min()}")
print(f"S1 with 0 candidates       : {(sizes==0).sum()}")
print("="*55)


MEMBER 2 — BLOCKING QUALITY REPORT
Total true matches         : 513
True matches in candidates : 513
Candidate Recall (micro)   : 1.0000
Candidate Recall (macro)   : 1.0000
Avg candidates / S1        : 401.71
Median candidates / S1     : 476
Max candidates / S1        : 551
Min candidates / S1        : 94
S1 with 0 candidates       : 0


### 8. Run on Test Data and Write Output

This section generates synthetic test data, applies the blocking process, and then writes the generated candidate pairs to a TSV file (`candidate_pairs.tsv`) in the `/content/output/` directory.

In [ ]:
# ------------------------------------------------------------
# 8. RUN ON TEST + WRITE candidate_pairs.tsv
# ------------------------------------------------------------
print("\nGenerating synthetic test data...")
test_s1, test_s2, test_s3, _ = make_synthetic_dataset(n_s1=200, seed=777)
test_s1 = prepare(test_s1); test_s2 = prepare(test_s2); test_s3 = prepare(test_s3)
test_others = pd.concat([test_s2, test_s3], ignore_index=True)

tnidx = build_name_indexes(test_others)
taidx = build_addr_indexes(test_others)
tcidx = build_combined_indexes(test_others)

print("Generating test candidates...")
test_cands = generate(test_s1, test_others, tnidx, taidx, tcidx,
                      top_k=20, fcut=65, tcut=0.20)

rows = [[sid, ",".join(sorted(test_cands.get(sid, set())))]
        for sid in test_s1["entity_id"]]
out = pd.DataFrame(rows, columns=["source1_entity_id","candidate_entity_ids"])
out.to_csv("/content/output/candidate_pairs.tsv", sep="\t", index=False)

print(f"\n✅ Wrote /content/output/candidate_pairs.tsv ({len(out)} rows)")
print("\nFirst 5 rows:")
print(out.head().to_string(index=False))


Generating synthetic test data...
Generating test candidates...

✅ Wrote /content/output/candidate_pairs.tsv (200 rows)

First 5 rows:
source1_entity_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

### 9. Validation

This section validates the generated `candidate_pairs.tsv` file to ensure it adheres to the expected format and contains valid entity IDs, checking for issues like incorrect columns, duplicate S1 rows, missing/extra S1 IDs, and malformed candidate IDs.

In [ ]:
# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------
df = pd.read_csv("/content/output/candidate_pairs.tsv",
                 sep="\t", dtype=str, keep_default_na=False)
issues = []
if list(df.columns) != ["source1_entity_id","candidate_entity_ids"]:
    issues.append(f"Bad columns: {df.columns.tolist()}")
if df["source1_entity_id"].duplicated().any():
    issues.append("Duplicate S1 rows")
expected = set(test_s1["entity_id"]); got = set(df["source1_entity_id"])
if expected != got:
    issues.append(f"Missing {len(expected-got)}, extra {len(got-expected)}")
valid_ids = set(test_others["entity_id"])
for _, r in df.iterrows():
    raw = r["candidate_entity_ids"]
    if not raw: continue
    ids = raw.split(",")
    if len(ids) != len(set(ids)):
        issues.append(f"Dup IDs in {r['source1_entity_id']}")
    for x in ids:
        if not (x.startswith("S2-") or x.startswith("S3-")):
            issues.append(f"Bad prefix {x}")
        elif x not in valid_ids:
            issues.append(f"Unknown ID {x}")

print("\n" + "="*55)
print("VALIDATION:", "✅ PASS" if not issues else "❌ FAIL")
print("="*55)
for m in issues[:10]: print("  -", m)
if not issues:
    print(f"  Rows              : {len(df)}")
    print(f"  Non-empty         : {(df['candidate_entity_ids'] != '').sum()}")
    print(f"  Empty (singletons): {(df['candidate_entity_ids'] == '').sum()}")


VALIDATION: ✅ PASS
  Rows              : 200
  Non-empty         : 200
  Empty (singletons): 0


### 10. Download Output

This final section provides a utility to download the generated `candidate_pairs.tsv` file from the Colab environment.

In [ ]:
# ------------------------------------------------------------
# 10. DOWNLOAD
# ------------------------------------------------------------
from google.colab import files
files.download("/content/output/candidate_pairs.tsv")
print("\n✅ Pipeline complete. File downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Pipeline complete. File downloaded.
